# &#128640; Growth Equestre | Caminho #1 - Lead Scoring com Dois Modelos

**Objetivo:** treinar **2 modelos candidatos** para propensao de qualificacao de leads, aplicar **GridSearchCV + fine tuning** e definir um **criterio profissional de desempate**.

## &#127919; Entregaveis deste notebook
- Modelo campeao (`best_model`)
- Modelo vice (`runner_up_model`)
- Relatorio de comparacao com metricas
- Criterio de desempate documentado para auditoria
- Artefatos exportados em `data/ml/artifacts/`


## &#129517; Estrategia de Modelagem

### Modelos candidatos
1. **Regressao Logistica** (baseline explicavel)
2. **Random Forest** (nao linear, robusto a interacoes)

### Metrica principal
- **ROC-AUC (validacao)**

### Criterio de desempate
Se a diferenca de ROC-AUC entre os 2 melhores for <= `0.005`, desempatar por:
1. maior **PR-AUC**
2. menor **Brier Score**
3. menor **latencia de inferencia**


In [1]:
import importlib.util
import subprocess
import sys

pkg_map = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",   # módulo -> pacote pip correto
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "jinja2": "jinja2",
}

missing = [pip_name for mod_name, pip_name in pkg_map.items()
           if importlib.util.find_spec(mod_name) is None]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

if missing:
    print("Instalando:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("OK")


OK


In [2]:
import pandas as pd

# Carregar o dataset
df = pd.read_csv("lead_scoring_dataset.csv")

# Verificar dimensões e primeiras linhas
print("Dimensões:", df.shape)
df.head()


Dimensões: (458, 13)


,lead_id,uf,cidade,segmento_interesse,orcamento_faixa,prazo_compra,status,n_events,n_page_view,n_hook_complete,n_cta_click,recency_last_event_hours,label_qualified
0,a484c6af-f02a-4509-a2b1-53a7f1647db4,GO,Rio Verde,EQUIPAMENTOS,60k+,90d,CURIOSO,2,1,1,0,1932.080053,0
1,80e25dfa-ffd0-4995-8d2f-4baf846035c5,SP,Sao Jose dos Campos,EQUIPAMENTOS,60k+,7d,ENVIADO,8,5,1,2,693.746719,1
2,447641a0-3e48-4e17-aa07-053b00bbfdbf,SP,Campinas,EVENTOS,20k-60k,90d,CURIOSO,1,1,0,0,1806.696719,0
3,2a18ab7f-fe0c-42d0-972d-d08b870676ed,SP,Sao Jose dos Campos,SERVICOS,5k-20k,7d,CURIOSO,2,2,0,0,268.780052,0
4,d4acd938-974e-45c2-a07c-e44c61fc6e47,GO,Rio Verde,SERVICOS,5k-20k,30d,CURIOSO,1,1,0,0,1536.696718,0


## &#128451;&#65039; Extracao do Dataset (Postgres -> CSV)

Esta celula gera um dataset tabular para treino com features de perfil + eventos do funil.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Carregar dataset
df = pd.read_csv("lead_scoring_dataset.csv")

# Conferir se todas as colunas importantes estão presentes
print(df.columns)

# Definir target
y = df["label_qualified"]

# Features: todas as colunas exceto o target e o id
X = df.drop(columns=["label_qualified", "lead_id"])

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape, "Teste:", X_test.shape)




Index(['lead_id', 'uf', 'cidade', 'segmento_interesse', 'orcamento_faixa',
       'prazo_compra', 'status', 'n_events', 'n_page_view', 'n_hook_complete',
       'n_cta_click', 'recency_last_event_hours', 'label_qualified'],
      dtype='object')
Treino: (366, 11) Teste: (92, 11)


In [4]:
# Bibliotecas padrao para IO e controle do fluxo de treino.
from pathlib import Path
import json
import time

# Persistencia de modelos e manipulacao tabular.
import joblib
import numpy as np
import pandas as pd

# Blocos principais do ecossistema sklearn para pipeline e avaliacao.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Configuracoes globais para reproducibilidade e padrao de artefatos.
RANDOM_STATE = 42
TARGET_COL = "label_qualified"
DATA_PATH = Path("data/ml/lead_scoring_dataset.csv")
ARTIFACT_DIR = Path("data/ml/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Facilita debug visual no notebook.
pd.set_option("display.max_columns", 200)


In [5]:
from pathlib import Path
import shutil

# Criar pasta
Path("data/ml").mkdir(parents=True, exist_ok=True)

# Mover arquivo para a pasta correta
shutil.copy("lead_scoring_dataset.csv", "data/ml/lead_scoring_dataset.csv")

# Agora funciona
DATA_PATH = Path("data/ml/lead_scoring_dataset.csv")
df = pd.read_csv(DATA_PATH)


In [6]:
# 1) Garantia de existencia do dataset.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado: {DATA_PATH}. Rode a celula de extracao SQL primeiro."
    )

# 2) Carrega o CSV de treino.
df = pd.read_csv(DATA_PATH)

# 3) Validacao de schema minimo esperado pelo pipeline.
required_cols = {
    "uf", "cidade", "segmento_interesse", "orcamento_faixa", "prazo_compra",
    "n_events", "n_page_view", "n_hook_complete", "n_cta_click", "recency_last_event_hours",
}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f"Colunas obrigatorias ausentes no dataset: {sorted(missing)}")

# 4) Se a label nao vier pronta, deriva pelo status comercial final.
if TARGET_COL not in df.columns:
    if "status" not in df.columns:
        raise ValueError("Dataset sem label_qualified e sem status para derivar o alvo.")
    df[TARGET_COL] = df["status"].astype(str).str.upper().isin(["QUALIFICADO", "ENVIADO"]).astype(int)

# 5) O treino supervisionado exige ao menos duas classes no alvo.
if df[TARGET_COL].nunique() < 2:
    raise ValueError(
        "Target com apenas uma classe. Gere mais dados de demo (QUALIFICADO/ENVIADO e CURIOSO/AQUECENDO)."
    )

# 6) Coerce de colunas numericas para garantir compatibilidade no pipeline.
for c in ["n_events", "n_page_view", "n_hook_complete", "n_cta_click", "recency_last_event_hours"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 7) Limpeza final: remove linhas sem target.
before = len(df)
df = df.dropna(subset=[TARGET_COL]).copy()
after = len(df)

# 8) Diagnostico rapido do dataset apos limpeza.
print(f"Linhas totais: {after} (removidas {before - after} sem target)")
print("Distribuicao de classe:")
print(df[TARGET_COL].value_counts(normalize=True).rename("ratio").round(4))

df.head(3)


Linhas totais: 458 (removidas 0 sem target)
Distribuicao de classe:
label_qualified
0    0.8013
1    0.1987
Name: ratio, dtype: float64


,lead_id,uf,cidade,segmento_interesse,orcamento_faixa,prazo_compra,status,n_events,n_page_view,n_hook_complete,n_cta_click,recency_last_event_hours,label_qualified
0,a484c6af-f02a-4509-a2b1-53a7f1647db4,GO,Rio Verde,EQUIPAMENTOS,60k+,90d,CURIOSO,2,1,1,0,1932.080053,0
1,80e25dfa-ffd0-4995-8d2f-4baf846035c5,SP,Sao Jose dos Campos,EQUIPAMENTOS,60k+,7d,ENVIADO,8,5,1,2,693.746719,1
2,447641a0-3e48-4e17-aa07-053b00bbfdbf,SP,Campinas,EVENTOS,20k-60k,90d,CURIOSO,1,1,0,0,1806.696719,0


## &#129504; Preparacao de Features e Split

- **Treino:** 70%
- **Validacao:** 15%
- **Teste:** 15%


In [7]:
# Definicao de grupos de features para facilitar manutencao.
numeric_features = [
    "n_events",
    "n_page_view",
    "n_hook_complete",
    "n_cta_click",
    "recency_last_event_hours",
]

categorical_features = [
    "uf",
    "cidade",
    "segmento_interesse",
    "orcamento_faixa",
    "prazo_compra",
]

# Seleciona X e y no formato esperado pelo sklearn.
feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int)

# Split 70/15/15 mantendo proporcao de classes (stratify).
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Shapes:")
print(f"  Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")

# Ajuste dinamico dos folds para evitar erro em datasets pequenos.
class_min_count = y_train.value_counts().min()
cv_splits = max(2, min(5, int(class_min_count)))
print(f"CV folds ajustado para: {cv_splits}")


Shapes:
  Train: (320, 10), Valid: (69, 10), Test: (69, 10)
CV folds ajustado para: 5


In [8]:
# Pipeline de preprocessamento para regressao logistica.
# Inclui scaling para estabilizar coeficientes em features numericas.
preprocess_logit = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Pipeline de preprocessamento para Random Forest.
# Nao precisa de scaling, mas mantem imputacao + one-hot.
preprocess_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Encadeia preprocessamento + estimador para busca de hiperparametros.
pipe_logit = Pipeline(
    steps=[
        ("prep", preprocess_logit),
        (
            "model",
            LogisticRegression(
                max_iter=2500,
                solver="liblinear",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipe_rf = Pipeline(
    steps=[
        ("prep", preprocess_rf),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Grade base para a 1a rodada de GridSearchCV.
param_grid_logit = {
    "model__C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"],
}

param_grid_rf = {
    "model__n_estimators": [200, 400, 700],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}

# CV estratificado para comparacao justa entre combinacoes de hiperparametros.
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)


In [9]:
# Instalar pacotes extras
!pip install xgboost lightgbm catboost pytorch-tabnet

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

# Pipeline para XGBoost
pipe_xgb = Pipeline(
    steps=[
        ("prep", preprocess_rf),  # mesmo preprocessamento do RF
        ("model", XGBClassifier(
            random_state=RANDOM_STATE,
            use_label_encoder=False,
            eval_metric="logloss"
        )),
    ]
)

param_grid_xgb = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 6, 10],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
}

# Pipeline para LightGBM
pipe_lgbm = Pipeline(
    steps=[
        ("prep", preprocess_rf),
        ("model", LGBMClassifier(random_state=RANDOM_STATE)),
    ]
)

param_grid_lgbm = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [-1, 10, 20],
    "model__learning_rate": [0.01, 0.1],
    "model__num_leaves": [31, 64],
}

# Pipeline para CatBoost
pipe_cat = Pipeline(
    steps=[
        ("prep", preprocess_rf),
        ("model", CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=0
        )),
    ]
)

param_grid_cat = {
    "model__iterations": [200, 400],
    "model__depth": [6, 10],
    "model__learning_rate": [0.01, 0.1],
}

# Pipeline para TabNet (opcional, mais pesado)
pipe_tabnet = Pipeline(
    steps=[
        ("prep", preprocess_rf),
        ("model", TabNetClassifier(
            seed=RANDOM_STATE,
            verbose=0
        )),
    ]
)

param_grid_tabnet = {
    "model__n_d": [8, 16],
    "model__n_a": [8, 16],
    "model__n_steps": [3, 5],
}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 55.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]


## &#128269; GridSearchCV (Treino Base)

Executa busca de hiperparametros para os 6 modelos usando **LogisticRegression; RandomForest; XGBoost; LightGBM; CatBoost; TabNet**.


In [10]:
def run_grid_search(name, pipeline, param_grid, X_train, y_train):
    # Wrapper para evitar repeticao e padronizar logging.
    print(f"\n>>> Treinando {name} com GridSearchCV...")
    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",  # metrica principal do projeto
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True,  # refit no melhor conjunto de hiperparametros
    )
    gs.fit(X_train, y_train)
    print(f"Melhor ROC-AUC CV ({name}): {gs.best_score_:.4f}")
    print("Melhores params:", gs.best_params_)
    return gs


# Rodada base (busca ampla) para todos os candidatos.
gs_logit = run_grid_search("LogisticRegression", pipe_logit, param_grid_logit, X_train, y_train)
gs_rf = run_grid_search("RandomForest", pipe_rf, param_grid_rf, X_train, y_train)
gs_xgb = run_grid_search("XGBoost", pipe_xgb, param_grid_xgb, X_train, y_train)
gs_lgbm = run_grid_search("LightGBM", pipe_lgbm, param_grid_lgbm, X_train, y_train)
gs_cat = run_grid_search("CatBoost", pipe_cat, param_grid_cat, X_train, y_train)
# Opcional (mais pesado)
gs_tabnet = run_grid_search("TabNet", pipe_tabnet, param_grid_tabnet, X_train, y_train)




>>> Treinando LogisticRegression com GridSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Melhor ROC-AUC CV (LogisticRegression): 0.9774
Melhores params: {'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l1'}

>>> Treinando RandomForest com GridSearchCV...
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Melhor ROC-AUC CV (RandomForest): 0.9805
Melhores params: {'model__class_weight': None, 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}

>>> Treinando XGBoost com GridSearchCV...
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:36:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Melhor ROC-AUC CV (XGBoost): 0.9847
Melhores params: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 400, 'model__subsample': 1.0}

>>> Treinando LightGBM com GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 64, number of negative: 256
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000047 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 181
[LightGBM] [Info] Number of data points in the train set: 320, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


Melhor ROC-AUC CV (TabNet): nan
Melhores params: {'model__n_a': 8, 'model__n_d': 8, 'model__n_steps': 3}


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


In [12]:
def evaluate_estimator(name, estimator, X_valid, y_valid, X_test, y_test):
    # Mede tempo de inferencia e coleta probabilidades em validacao.
    t0 = time.perf_counter()
    p_valid = estimator.predict_proba(X_valid)[:, 1]
    latency_valid = (time.perf_counter() - t0) * 1000 / max(1, len(X_valid))

    # Repete medicao em teste para comparacao de estabilidade.
    t1 = time.perf_counter()
    p_test = estimator.predict_proba(X_test)[:, 1]
    latency_test = (time.perf_counter() - t1) * 1000 / max(1, len(X_test))

    # Threshold padrao para metricas baseadas em classe predita.
    yhat_valid = (p_valid >= 0.5).astype(int)
    yhat_test = (p_test >= 0.5).astype(int)

    return {
        "model": name,
        "val_roc_auc": roc_auc_score(y_valid, p_valid),
        "val_pr_auc": average_precision_score(y_valid, p_valid),
        "val_brier": brier_score_loss(y_valid, p_valid),
        "val_f1": f1_score(y_valid, yhat_valid, zero_division=0),
        "val_precision": precision_score(y_valid, yhat_valid, zero_division=0),
        "val_recall": recall_score(y_valid, yhat_valid, zero_division=0),
        "val_latency_ms": latency_valid,
        "test_roc_auc": roc_auc_score(y_test, p_test),
        "test_pr_auc": average_precision_score(y_test, p_test),
        "test_brier": brier_score_loss(y_test, p_test),
        "test_f1": f1_score(y_test, yhat_test, zero_division=0),
        "test_precision": precision_score(y_test, yhat_test, zero_division=0),
        "test_recall": recall_score(y_test, yhat_test, zero_division=0),
        "test_latency_ms": latency_test,
    }


def select_winner(results_df, eps_auc=0.005, eps_pr=0.003, eps_brier=0.002):
    # Ordena por metricas principais para iniciar desempate.
    ranked = results_df.sort_values(["val_roc_auc", "val_pr_auc"], ascending=[False, False]).reset_index(drop=True)
    a = ranked.iloc[0]
    b = ranked.iloc[1]

    reasons = []

    # Regra 1: ROC-AUC.
    if (a["val_roc_auc"] - b["val_roc_auc"]) > eps_auc:
        reasons.append("ROC-AUC superior sem empate técnico.")
        return a["model"], reasons

    # Regra 2: PR-AUC.
    reasons.append("Empate técnico em ROC-AUC; aplicando desempate por PR-AUC.")
    if (a["val_pr_auc"] - b["val_pr_auc"]) > eps_pr:
        reasons.append("PR-AUC decidiu o vencedor.")
        return a["model"], reasons

    # Regra 3: Brier Score (menor e melhor).
    reasons.append("Empate técnico em PR-AUC; aplicando desempate por Brier Score.")
    if (b["val_brier"] - a["val_brier"]) > eps_brier:
        reasons.append("Brier Score decidiu o vencedor (menor é melhor).")
        return a["model"], reasons

    # Regra 4: Latência média de inferência.
    reasons.append("Empate técnico em Brier; aplicando desempate por latência.")
    if a["val_latency_ms"] <= b["val_latency_ms"]:
        reasons.append("Latência de inferência decidiu o vencedor.")
        return a["model"], reasons

    reasons.append("Latência decidiu o vencedor (modelo B).")
    return b["model"], reasons

# Avaliação dos modelos no conjunto de validação e teste
results = []

for name, gs in [
    ("LogisticRegression", gs_logit),
    ("RandomForest", gs_rf),
    ("XGBoost", gs_xgb),
    ("LightGBM", gs_lgbm),
    ("CatBoost", gs_cat),
    ("TabNet", gs_tabnet),
]:
    est = gs.best_estimator_
    results.append(evaluate_estimator(name, est, X_valid, y_valid, X_test, y_test))

results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)

# Seleciona campeão e vice
winner_name, winner_reasons = select_winner(results_df)
runner_up_name = [m for m in results_df["model"] if m != winner_name][0]

print("Resultados comparativos:")
display(results_df)

print(f"\n🏆 Campeão: {winner_name}")
print(f"🥈 Vice: {runner_up_name}")
print("Razões da escolha:")
for r in winner_reasons:
    print("-", r)


Resultados comparativos:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,val_roc_auc,val_pr_auc,val_brier,val_f1,val_precision,val_recall,val_latency_ms,test_roc_auc,test_pr_auc,test_brier,test_f1,test_precision,test_recall,test_latency_ms
0,CatBoost,0.972727,0.888040,0.067695,0.695652,0.888889,0.571429,0.149835,1.000000,1.000000,0.012064,1.000000,1.000000,1.0,0.116266
1,XGBoost,0.972727,0.895320,0.059511,0.750000,0.900000,0.642857,0.148346,1.000000,1.000000,0.016130,0.928571,0.866667,1.0,0.139768
2,LogisticRegression,0.967532,0.818432,0.065301,0.695652,0.888889,0.571429,0.156408,1.000000,1.000000,0.024976,1.000000,1.000000,1.0,0.132551
3,LightGBM,0.962987,0.819717,0.071316,0.695652,0.888889,0.571429,0.143397,1.000000,1.000000,0.019359,0.962963,0.928571,1.0,0.131956
4,RandomForest,0.959740,0.844241,0.073611,0.695652,0.888889,0.571429,1.110231,1.000000,1.000000,0.016229,1.000000,1.000000,1.0,0.922368
5,TabNet,0.538961,0.217153,0.192380,0.000000,0.000000,0.000000,0.371471,0.474588,0.276149,0.171488,0.000000,0.000000,0.0,0.283384



🏆 Campeão: XGBoost
🥈 Vice: CatBoost
Razões da escolha:
- Empate técnico em ROC-AUC; aplicando desempate por PR-AUC.
- PR-AUC decidiu o vencedor.


## &#127912; Fine Tuning (2a rodada)

Refina o espaco de busca ao redor dos melhores parametros encontrados na rodada base.


In [16]:
def build_fine_grid_logit(best_params):
    # Refina C ao redor do melhor valor da rodada base.
    c = float(best_params["model__C"])
    c_candidates = sorted({max(1e-4, round(v, 5)) for v in [c * 0.5, c * 0.75, c, c * 1.25, c * 1.5]})
    return {
        "model__C": c_candidates,
        "model__penalty": [best_params["model__penalty"]],
        "model__class_weight": [best_params["model__class_weight"], "balanced"],
    }

def build_fine_grid_rf(best_params):
    # Refina floresta ao redor do melhor ponto base.
    n_estimators = int(best_params["model__n_estimators"])
    max_depth = best_params["model__max_depth"]
    min_split = int(best_params["model__min_samples_split"])
    min_leaf = int(best_params["model__min_samples_leaf"])

    n_estimators_candidates = sorted({max(100, n_estimators - 150), n_estimators, n_estimators + 150})

    depth_candidates = [max_depth]
    if isinstance(max_depth, int):
        depth_candidates = sorted({max(3, max_depth - 4), max_depth, max_depth + 4})

    return {
        "model__n_estimators": n_estimators_candidates,
        "model__max_depth": depth_candidates,
        "model__min_samples_split": sorted({max(2, min_split - 1), min_split, min_split + 1}),
        "model__min_samples_leaf": sorted({max(1, min_leaf - 1), min_leaf, min_leaf + 1}),
        "model__class_weight": [best_params["model__class_weight"], "balanced_subsample"],
    }

def build_fine_grid_xgb(best_params):
    # Refina hiperparâmetros ao redor do melhor ponto base.
    n_estimators = int(best_params["model__n_estimators"])
    max_depth = int(best_params["model__max_depth"])
    lr = float(best_params["model__learning_rate"])

    return {
        "model__n_estimators": [max(100, n_estimators - 100), n_estimators, n_estimators + 100],
        "model__max_depth": [max(3, max_depth - 2), max_depth, max_depth + 2],
        "model__learning_rate": [round(lr * 0.5, 3), lr, round(lr * 1.5, 3)],
        "model__subsample": [best_params["model__subsample"]],
    }

def build_fine_grid_lgbm(best_params):
    n_estimators = int(best_params["model__n_estimators"])
    num_leaves = int(best_params["model__num_leaves"])
    lr = float(best_params["model__learning_rate"])

    return {
        "model__n_estimators": [max(100, n_estimators - 100), n_estimators, n_estimators + 100],
        "model__num_leaves": [max(16, num_leaves - 16), num_leaves, num_leaves + 16],
        "model__learning_rate": [round(lr * 0.5, 3), lr, round(lr * 1.5, 3)],
        "model__max_depth": [best_params["model__max_depth"]],
    }

def build_fine_grid_cat(best_params):
    iterations = int(best_params["model__iterations"])
    depth = int(best_params["model__depth"])
    lr = float(best_params["model__learning_rate"])

    return {
        "model__iterations": [max(100, iterations - 100), iterations, iterations + 100],
        "model__depth": [max(4, depth - 2), depth, depth + 2],
        "model__learning_rate": [round(lr * 0.5, 3), lr, round(lr * 1.5, 3)],
    }

def build_fine_grid_tabnet(best_params):
    # Refina TabNet ao redor do melhor ponto base.
    n_d = int(best_params["model__n_d"])
    n_a = int(best_params["model__n_a"])
    n_steps = int(best_params["model__n_steps"])

    return {
        "model__n_d": [n_d // 2, n_d, n_d * 2] if n_d > 1 else [n_d, n_d * 2],
        "model__n_a": [n_a // 2, n_a, n_a * 2] if n_a > 1 else [n_a, n_a * 2],
        "model__n_steps": [max(1, n_steps - 1), n_steps, n_steps + 1],
    }

# Grades refinadas da 2a rodada
fine_grid_logit = build_fine_grid_logit(gs_logit.best_params_)
fine_grid_rf = build_fine_grid_rf(gs_rf.best_params_)
fine_grid_xgb = build_fine_grid_xgb(gs_xgb.best_params_)
fine_grid_lgbm = build_fine_grid_lgbm(gs_lgbm.best_params_)
fine_grid_cat = build_fine_grid_cat(gs_cat.best_params_)
fine_grid_tabnet = build_fine_grid_tabnet(gs_tabnet.best_params_)

# Rodada de fine tuning
fine_logit = run_grid_search("LogisticRegression (fine)", pipe_logit, fine_grid_logit, X_train, y_train)
fine_rf = run_grid_search("RandomForest (fine)", pipe_rf, fine_grid_rf, X_train, y_train)
fine_xgb = run_grid_search("XGBoost (fine)", pipe_xgb, fine_grid_xgb, X_train, y_train)
fine_lgbm = run_grid_search("LightGBM (fine)", pipe_lgbm, fine_grid_lgbm, X_train, y_train)
fine_cat = run_grid_search("CatBoost (fine)", pipe_cat, fine_grid_cat, X_train, y_train)
fine_tabnet = run_grid_search("TabNet (fine)", pipe_tabnet, fine_grid_tabnet, X_train, y_train)



>>> Treinando LogisticRegression (fine) com GridSearchCV...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Melhor ROC-AUC CV (LogisticRegression (fine)): 0.9774
Melhores params: {'model__C': 0.05, 'model__class_weight': None, 'model__penalty': 'l1'}

>>> Treinando RandomForest (fine) com GridSearchCV...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Melhor ROC-AUC CV (RandomForest (fine)): 0.9805
Melhores params: {'model__class_weight': None, 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}

>>> Treinando XGBoost (fine) com GridSearchCV...
Fitting 5 folds for each of 27 candidates, totalling 135 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:57:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Melhor ROC-AUC CV (XGBoost (fine)): 0.9863
Melhores params: {'model__learning_rate': 0.015, 'model__max_depth': 3, 'model__n_estimators': 500, 'model__subsample': 1.0}

>>> Treinando LightGBM (fine) com GridSearchCV...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[LightGBM] [Info] Number of positive: 64, number of negative: 256
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000035 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 181
[LightGBM] [Info] Number of data points in the train set: 320, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Melhor ROC-AUC CV (TabNet (fine)): nan
Melhores params: {'model__n_a': 4, 'model__n_d': 4, 'model__n_steps': 2}


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


## &#128202; Avaliacao e Criterio de Desempate

Regra implementada:
- 1o: maior **ROC-AUC validacao**
- empate tecnico (<= 0.005): maior **PR-AUC validacao**
- persistindo empate: menor **Brier validacao**
- persistindo empate: menor **latencia (ms/registro)**


In [17]:
def evaluate_estimator(name, estimator, X_valid, y_valid, X_test, y_test):
    # Mede tempo de inferencia e coleta probabilidades em validacao.
    t0 = time.perf_counter()
    p_valid = estimator.predict_proba(X_valid)[:, 1]
    latency_valid = (time.perf_counter() - t0) * 1000 / max(1, len(X_valid))

    # Repete medicao em teste para comparacao de estabilidade.
    t1 = time.perf_counter()
    p_test = estimator.predict_proba(X_test)[:, 1]
    latency_test = (time.perf_counter() - t1) * 1000 / max(1, len(X_test))

    # Threshold padrao para metricas baseadas em classe predita.
    yhat_valid = (p_valid >= 0.5).astype(int)
    yhat_test = (p_test >= 0.5).astype(int)

    return {
        "model": name,
        "val_roc_auc": roc_auc_score(y_valid, p_valid),
        "val_pr_auc": average_precision_score(y_valid, p_valid),
        "val_brier": brier_score_loss(y_valid, p_valid),
        "val_f1": f1_score(y_valid, yhat_valid, zero_division=0),
        "val_precision": precision_score(y_valid, yhat_valid, zero_division=0),
        "val_recall": recall_score(y_valid, yhat_valid, zero_division=0),
        "val_latency_ms": latency_valid,
        "test_roc_auc": roc_auc_score(y_test, p_test),
        "test_pr_auc": average_precision_score(y_test, p_test),
        "test_brier": brier_score_loss(y_test, p_test),
        "test_f1": f1_score(y_test, yhat_test, zero_division=0),
        "test_precision": precision_score(y_test, yhat_test, zero_division=0),
        "test_recall": recall_score(y_test, yhat_test, zero_division=0),
        "test_latency_ms": latency_test,
    }


def select_winner(results_df, eps_auc=0.005, eps_pr=0.003, eps_brier=0.002):
    # Ordena por metricas principais para iniciar desempate.
    ranked = results_df.sort_values(["val_roc_auc", "val_pr_auc"], ascending=[False, False]).reset_index(drop=True)
    a = ranked.iloc[0]
    b = ranked.iloc[1]

    reasons = []

    # Regra 1: ROC-AUC.
    if (a["val_roc_auc"] - b["val_roc_auc"]) > eps_auc:
        reasons.append("ROC-AUC superior sem empate técnico.")
        return a["model"], reasons

    # Regra 2: PR-AUC.
    reasons.append("Empate técnico em ROC-AUC; aplicando desempate por PR-AUC.")
    if (a["val_pr_auc"] - b["val_pr_auc"]) > eps_pr:
        reasons.append("PR-AUC decidiu o vencedor.")
        return a["model"], reasons

    # Regra 3: Brier Score (menor e melhor).
    reasons.append("Empate técnico em PR-AUC; aplicando desempate por Brier Score.")
    if (b["val_brier"] - a["val_brier"]) > eps_brier:
        reasons.append("Brier Score decidiu o vencedor (menor é melhor).")
        return a["model"], reasons

    # Regra 4: Latência média de inferência.
    reasons.append("Empate técnico em Brier; aplicando desempate por latência.")
    if a["val_latency_ms"] <= b["val_latency_ms"]:
        reasons.append("Latência de inferência decidiu o vencedor.")
        return a["model"], reasons

    reasons.append("Latência decidiu o vencedor (modelo B).")
    return b["model"], reasons


# Avalia todos os modelos da rodada de fine tuning
results = [
    evaluate_estimator("logit_fine", fine_logit.best_estimator_, X_valid, y_valid, X_test, y_test),
    evaluate_estimator("rf_fine", fine_rf.best_estimator_, X_valid, y_valid, X_test, y_test),
    evaluate_estimator("xgb_fine", fine_xgb.best_estimator_, X_valid, y_valid, X_test, y_test),
    evaluate_estimator("lgbm_fine", fine_lgbm.best_estimator_, X_valid, y_valid, X_test, y_test),
    evaluate_estimator("cat_fine", fine_cat.best_estimator_, X_valid, y_valid, X_test, y_test),
    evaluate_estimator("tabnet_fine", fine_tabnet.best_estimator_, X_valid, y_valid, X_test, y_test),
]

# Cria DataFrame com resultados ordenados
results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)

# Seleciona campeão e vice aplicando critério de desempate
winner_name, winner_reasons = select_winner(results_df)
runner_up_name = results_df.iloc[1]["model"]
third_place_name = results_df.iloc[2]["model"]

print("Resultados comparativos (fine tuning):")
display(results_df)

print(f"\n🏆 1º lugar (Campeão): {winner_name}")
print(f"🥈 2º lugar (Vice): {runner_up_name}")
print(f"🥉 3º lugar: {third_place_name}")
print("\nRazões da escolha do campeão:")
for r in winner_reasons:
    print("-", r)


Resultados comparativos (fine tuning):


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,val_roc_auc,val_pr_auc,val_brier,val_f1,val_precision,val_recall,val_latency_ms,test_roc_auc,test_pr_auc,test_brier,test_f1,test_precision,test_recall,test_latency_ms
0,xgb_fine,0.979221,0.911052,0.055898,0.769231,0.833333,0.714286,0.149308,1.000000,1.000000,0.022682,0.896552,0.812500,1.0,0.137647
1,logit_fine,0.967532,0.818432,0.069498,0.695652,0.888889,0.571429,0.122724,1.000000,1.000000,0.038144,1.000000,1.000000,1.0,0.109308
2,cat_fine,0.967532,0.872496,0.069239,0.695652,0.888889,0.571429,0.134231,1.000000,1.000000,0.019655,1.000000,1.000000,1.0,0.120595
3,lgbm_fine,0.962987,0.819717,0.071316,0.695652,0.888889,0.571429,0.145087,1.000000,1.000000,0.019359,0.962963,0.928571,1.0,0.126462
4,rf_fine,0.959740,0.844241,0.073611,0.695652,0.888889,0.571429,1.086700,1.000000,1.000000,0.016229,1.000000,1.000000,1.0,1.081622
5,tabnet_fine,0.350000,0.160484,0.568744,0.337349,0.202899,1.000000,0.317404,0.628434,0.460134,0.630300,0.317073,0.188406,1.0,0.280310



🏆 1º lugar (Campeão): xgb_fine
🥈 2º lugar (Vice): logit_fine
🥉 3º lugar: cat_fine

Razões da escolha do campeão:
- ROC-AUC superior sem empate técnico.


In [18]:
# Tabela com realce visual para facilitar leitura de decisão no pitch.
display(
    results_df.style
    .format({
        "val_roc_auc": "{:.4f}",
        "val_pr_auc": "{:.4f}",
        "val_brier": "{:.4f}",
        "val_f1": "{:.4f}",
        "val_precision": "{:.4f}",
        "val_recall": "{:.4f}",
        "val_latency_ms": "{:.4f}",
        "test_roc_auc": "{:.4f}",
        "test_pr_auc": "{:.4f}",
        "test_brier": "{:.4f}",
        "test_f1": "{:.4f}",
        "test_precision": "{:.4f}",
        "test_recall": "{:.4f}",
        "test_latency_ms": "{:.4f}",
    })
    # Verde: quanto maior melhor.
    .background_gradient(subset=["val_roc_auc", "val_pr_auc", "test_roc_auc", "test_pr_auc"], cmap="Greens")
    # Vermelho invertido: quanto menor melhor (brier e latência).
    .background_gradient(subset=["val_brier", "test_brier", "val_latency_ms", "test_latency_ms"], cmap="OrRd_r")
)

# Exibe ranking com 1º, 2º e 3º lugar
print(f"[🏆 WINNER] 1º lugar: {winner_name}")
print(f"[🥈 RUNNER-UP] 2º lugar: {runner_up_name}")
print(f"[🥉 THIRD PLACE] 3º lugar: {third_place_name}")

print("\nRazões da escolha do campeão:")
for r in winner_reasons:
    print(f"  - {r}")



,model,val_roc_auc,val_pr_auc,val_brier,val_f1,val_precision,val_recall,val_latency_ms,test_roc_auc,test_pr_auc,test_brier,test_f1,test_precision,test_recall,test_latency_ms
0,xgb_fine,0.9792,0.9111,0.0559,0.7692,0.8333,0.7143,0.1493,1.0000,1.0000,0.0227,0.8966,0.8125,1.0000,0.1376
1,logit_fine,0.9675,0.8184,0.0695,0.6957,0.8889,0.5714,0.1227,1.0000,1.0000,0.0381,1.0000,1.0000,1.0000,0.1093
2,cat_fine,0.9675,0.8725,0.0692,0.6957,0.8889,0.5714,0.1342,1.0000,1.0000,0.0197,1.0000,1.0000,1.0000,0.1206
3,lgbm_fine,0.9630,0.8197,0.0713,0.6957,0.8889,0.5714,0.1451,1.0000,1.0000,0.0194,0.9630,0.9286,1.0000,0.1265
4,rf_fine,0.9597,0.8442,0.0736,0.6957,0.8889,0.5714,1.0867,1.0000,1.0000,0.0162,1.0000,1.0000,1.0000,1.0816
5,tabnet_fine,0.3500,0.1605,0.5687,0.3373,0.2029,1.0000,0.3174,0.6284,0.4601,0.6303,0.3171,0.1884,1.0000,0.2803


[🏆 WINNER] 1º lugar: xgb_fine
[🥈 RUNNER-UP] 2º lugar: logit_fine
[🥉 THIRD PLACE] 3º lugar: cat_fine

Razões da escolha do campeão:
  - ROC-AUC superior sem empate técnico.


## &#128190; Persistencia dos Artefatos

Salva campeao, vice e relatorio de selecao para integracao com API de scoring.


In [20]:
# Mapeia nomes para os estimadores finais treinados (fine tuning)
model_map = {
    "logit_fine": fine_logit.best_estimator_,
    "rf_fine": fine_rf.best_estimator_,
    "xgb_fine": fine_xgb.best_estimator_,
    "lgbm_fine": fine_lgbm.best_estimator_,
    "cat_fine": fine_cat.best_estimator_,
    "tabnet_fine": fine_tabnet.best_estimator_,
}

# Define campeão e vice para uso operacional (champion/challenger).
best_model = model_map[winner_name]
runner_up_name = [m for m in model_map.keys() if m != winner_name][0]
runner_up_model = model_map[runner_up_name]

# Persistência de modelos em joblib para inferência no scoring_service.
joblib.dump(best_model, ARTIFACT_DIR / "lead_scoring_best_model.joblib")
joblib.dump(runner_up_model, ARTIFACT_DIR / "lead_scoring_runner_up_model.joblib")

# Relatório para auditoria e reprodução da decisão de modelo.
report = {
    "winner": winner_name,
    "runner_up": runner_up_name,
    "third_place": third_place_name,  # agora também incluímos o 3º lugar
    "selection_reasons": winner_reasons,
    "metrics": results_df.to_dict(orient="records"),
    "random_state": RANDOM_STATE,
    "target_col": TARGET_COL,
}

with open(ARTIFACT_DIR / "model_selection_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("[OK] Artefatos salvos com sucesso:")
print(f" - {ARTIFACT_DIR / 'lead_scoring_best_model.joblib'}")
print(f" - {ARTIFACT_DIR / 'lead_scoring_runner_up_model.joblib'}")
print(f" - {ARTIFACT_DIR / 'model_selection_report.json'}")



[OK] Artefatos salvos com sucesso:
 - data/ml/artifacts/lead_scoring_best_model.joblib
 - data/ml/artifacts/lead_scoring_runner_up_model.joblib
 - data/ml/artifacts/model_selection_report.json


In [21]:
%%bash
set -euo pipefail

echo "Resumo dos artefatos"
ls -lh data/ml/artifacts

echo ""
echo "Conteúdo do relatório de seleção:"
cat data/ml/artifacts/model_selection_report.json



Resumo dos artefatos
total 484K
-rw-r--r-- 1 root root 471K Feb 24 16:03 lead_scoring_best_model.joblib
-rw-r--r-- 1 root root 5.9K Feb 24 16:03 lead_scoring_runner_up_model.joblib
-rw-r--r-- 1 root root 3.6K Feb 24 16:03 model_selection_report.json

Conteúdo do relatório de seleção:
{
  "winner": "xgb_fine",
  "runner_up": "logit_fine",
  "third_place": "cat_fine",
  "selection_reasons": [
    "ROC-AUC superior sem empate técnico."
  ],
  "metrics": [
    {
      "model": "xgb_fine",
      "val_roc_auc": 0.9792207792207792,
      "val_pr_auc": 0.9110521539618177,
      "val_brier": 0.055898180865297915,
      "val_f1": 0.7692307692307693,
      "val_precision": 0.8333333333333334,
      "val_recall": 0.7142857142857143,
      "val_latency_ms": 0.14930844928012052,
      "test_roc_auc": 1.0,
      "test_pr_auc": 0.9999999999999998,
      "test_brier": 0.02268241130779709,
      "test_f1": 0.896551724137931,
      "test_precision": 0.8125,
      "test_recall": 1.0,
      "test_latency_m

📌 Sobre os resultados
O fato de os modelos novos terem superado os anteriores não significa que eles sejam melhores.

Isso indica que, com o volume de dados atual, os modelos mais modernos não conseguiram extrair padrões adicionais além do que já estava sendo capturado pelo Logit e pelo Random Forest.

📊 O que podemos inferir
Pouco volume de dados: modelos como XGBoost, LightGBM e CatBoost brilham quando há muitos exemplos e variabilidade. Com datasets pequenos, eles podem não mostrar vantagem significativa.

Ausência de overfitting (“vício”): o fato de os modelos diferentes atingirem métricas semelhantes mostra que o problema está bem definido e que não houve “vicio” ou sobreajuste. Se houvesse overfitting, veríamos métricas muito boas em treino mas ruins em validação/teste.

Consistência entre modelos: vários algoritmos diferentes chegando em métricas parecidas é um bom sinal de robustez — significa que o dataset está representando bem o fenômeno e que não depende de um único modelo para funcionar.

✅ Conclusão

O dataset pequeno limita uma avaliação mais precisa dos modelos mais modernos.

O fato de todos convergirem para métricas semelhantes mostra que não houve vício e que o pipeline está saudável.

Isso é positivo: Com esses modelos treinados nos temos uma base sólida para quando aumentar o volume de dados, aí os modelos mais sofisticados podem contribuir para manter a estrutura bem treinada.

## &#128268; Proximo Passo de Integracao

1. Copiar `lead_scoring_best_model.joblib` para o servico de scoring
2. Atualizar o endpoint `/score` para consumir o modelo campeao
3. Manter o `runner_up_model` como fallback tecnico
4. Versionar o JSON de selecao para auditoria no pitch


## ⚠️ 🚨 As células abaixo só se aplicam dentro do projeto. Aqui no Colab elas mão tem nenhuma função.

## &#9203; Execução CLI (sem notebook)

Se preferir rodar fora do Jupyter, use o script:

```bash
python tools/ml/train_lead_scoring.py --input-csv data/ml/lead_scoring_dataset.csv --output-dir data/ml/artifacts
```

📌 célula — Importações e configuração

In [ ]:
from pathlib import Path
import subprocess
import sys

# Ajusta raiz do repo automaticamente
cwd = Path.cwd()
repo_root = cwd
for p in [cwd, *cwd.parents]:
    if (p / "tools" / "ml" / "train_lead_scoring.py").exists():
        repo_root = p
        break


In [ ]:
from pathlib import Path
import subprocess
import sys

# Ajusta raiz do repo automaticamente
cwd = Path.cwd()
repo_root = cwd
for p in [cwd, *cwd.parents]:
    if (p / "tools" / "ml" / "train_lead_scoring.py").exists():
        repo_root = p
        break

cmd = [
    sys.executable,
    str(repo_root / "tools" / "ml" / "train_lead_scoring.py"),
    "--input-csv",
    str(repo_root / "data" / "ml" / "lead_scoring_dataset.csv"),
    "--output-dir",
    str(repo_root / "data" / "ml" / "artifacts"),
]

print("Executando:", " ".join(cmd))
subprocess.check_call(cmd, cwd=repo_root)
print("Treino via CLI concluido com sucesso.")
